In [3]:
import pandas as pd
import numpy as np
import joblib
import torch
import torch.nn as nn
import os
from sklearn.neighbors import NearestNeighbors
import warnings
warnings.filterwarnings('ignore')

# Load the processed data and mappings
df_features = pd.read_parquet('../data/processed/processed_features.parquet')
track_mapping = joblib.load('../saved_models/preprocessors/track_mapping.pkl')
trackid_to_index = joblib.load('../saved_models/preprocessors/trackid_to_index.pkl')

# Separate track_id from the actual audio features
track_ids = df_features['track_id'].values
X = df_features.drop(columns=['track_id']).values 

print(f"Data loaded. Shape: {X.shape}")

Data loaded. Shape: (89741, 33)


In [4]:
print("=== Training Custom Baseline Model ===")
# NearestNeighbors is highly optimized and avoids creating a massive 80k x 80k matrix in memory
baseline_model = NearestNeighbors(n_neighbors=11, metric='cosine', algorithm='brute', n_jobs=-1)
baseline_model.fit(X)

# Save the baseline model
os.makedirs('../saved_models', exist_ok=True)
joblib.dump(baseline_model, '../saved_models/baseline_nn_model.pkl')
print("Baseline Model trained and saved to '../saved_models/baseline_nn_model.pkl'")

=== Training Custom Baseline Model ===
Baseline Model trained and saved to '../saved_models/baseline_nn_model.pkl'


In [9]:
print("=== Building Stable RBM (Autoencoder) Architecture ===")

class RBM(nn.Module):
    def __init__(self, n_visible, n_hidden):
        super(RBM, self).__init__()
        # Encoder: Compresses visible features into the hidden "latent" space
        self.encoder = nn.Linear(n_visible, n_hidden)
        # Decoder: Reconstructs the features back from the hidden space
        self.decoder = nn.Linear(n_hidden, n_visible)

    def forward(self, x):
        # Pass through encoder and apply sigmoid to keep values between 0 and 1
        hidden = torch.sigmoid(self.encoder(x))
        # Pass through decoder to reconstruct
        reconstructed = torch.sigmoid(self.decoder(hidden))
        return reconstructed, hidden

# Hyperparameters
n_visible = X.shape[1]
n_hidden = 64 # The size of our "latent" musical essence space
batch_size = 256
epochs = 20
learning_rate = 0.001

# Initialize Model, Loss Function, and Optimizer
rbm_model = RBM(n_visible, n_hidden)
criterion = nn.MSELoss() # Mean Squared Error for reconstruction
optimizer = torch.optim.Adam(rbm_model.parameters(), lr=learning_rate)

print(f"Model initialized. Visible: {n_visible} -> Hidden: {n_hidden}")

=== Building Stable RBM (Autoencoder) Architecture ===
Model initialized. Visible: 33 -> Hidden: 64


In [10]:
print("=== Training RBM ===")
rbm_model.train()

for epoch in range(epochs):
    epoch_loss = 0
    num_batches = 0
    
    # Shuffle data
    indices = torch.randperm(X_tensor.size(0))
    X_shuffled = X_tensor[indices]
    
    for i in range(0, X_shuffled.size(0), batch_size):
        batch = X_shuffled[i:i+batch_size]
        
        # 1. Zero the gradients
        optimizer.zero_grad()
        
        # 2. Forward pass
        v_rec, h = rbm_model(batch)
        
        # 3. Calculate loss (How different is the reconstruction from the original?)
        loss = criterion(v_rec, batch)
        
        # 4. Backward pass and optimize
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        num_batches += 1
        
    # Print the AVERAGE loss per batch for this epoch
    avg_loss = epoch_loss / num_batches
    print(f"Epoch {epoch+1}/{epochs} - Average Loss: {avg_loss:.4f}")

print("RBM Training Complete!")

=== Training RBM ===
Epoch 1/20 - Average Loss: 0.0703
Epoch 2/20 - Average Loss: 0.0439
Epoch 3/20 - Average Loss: 0.0277
Epoch 4/20 - Average Loss: 0.0141
Epoch 5/20 - Average Loss: 0.0076
Epoch 6/20 - Average Loss: 0.0054
Epoch 7/20 - Average Loss: 0.0042
Epoch 8/20 - Average Loss: 0.0031
Epoch 9/20 - Average Loss: 0.0024
Epoch 10/20 - Average Loss: 0.0019
Epoch 11/20 - Average Loss: 0.0015
Epoch 12/20 - Average Loss: 0.0013
Epoch 13/20 - Average Loss: 0.0011
Epoch 14/20 - Average Loss: 0.0010
Epoch 15/20 - Average Loss: 0.0008
Epoch 16/20 - Average Loss: 0.0007
Epoch 17/20 - Average Loss: 0.0007
Epoch 18/20 - Average Loss: 0.0006
Epoch 19/20 - Average Loss: 0.0006
Epoch 20/20 - Average Loss: 0.0006
RBM Training Complete!


In [11]:
print("=== Generating RBM Hidden Representations ===")
rbm_model.eval()

with torch.no_grad():
    # Pass all data through the model to get the hidden (latent) representations
    _, hidden_representations = rbm_model(X_tensor)

# Convert to numpy for sklearn
X_hidden = hidden_representations.numpy()

# Train a NearestNeighbors model on this new, compressed "latent" space
rbm_nn_model = NearestNeighbors(n_neighbors=11, metric='cosine', algorithm='brute', n_jobs=-1)
rbm_nn_model.fit(X_hidden)

# Save the trained PyTorch model and the NearestNeighbors model
torch.save(rbm_model.state_dict(), '../saved_models/rbm_weights.pth')
joblib.dump(rbm_nn_model, '../saved_models/rbm_nn_model.pkl')

print("Models saved successfully to '../saved_models/'")

=== Generating RBM Hidden Representations ===
Models saved successfully to '../saved_models/'


In [12]:
# Pick a random track to test
test_track_id = track_ids[50] 
idx = trackid_to_index[test_track_id]

print(f"\n--- Testing on: {track_mapping[test_track_id]['track_name']} by {track_mapping[test_track_id]['artists']} ---\n")

# Test Baseline
distances, indices = baseline_model.kneighbors(X[idx].reshape(1, -1))
print("1. Baseline (Content-Based) Recommendations:")
for i in indices[0][1:4]: # Show top 3
    rec_id = track_ids[i]
    print(f"   - {track_mapping[rec_id]['track_name']} ({track_mapping[rec_id]['track_genre']})")

# Test RBM
distances, indices = rbm_nn_model.kneighbors(X_hidden[idx].reshape(1, -1))
print("\n2. RBM (Latent Space) Recommendations:")
for i in indices[0][1:4]: # Show top 3
    rec_id = track_ids[i]
    print(f"   - {track_mapping[rec_id]['track_name']} ({track_mapping[rec_id]['track_genre']})")


--- Testing on: Trampoline - Acoustic by Highland Peak ---

1. Baseline (Content-Based) Recommendations:
   - (I Just) Died In Your Arms - Acoustic (acoustic)
   - Hey There Delilah (acoustic)
   - Dance Monkey (acoustic)

2. RBM (Latent Space) Recommendations:
   - Hey There Delilah (acoustic)
   - (I Just) Died In Your Arms - Acoustic (acoustic)
   - Dance Monkey (acoustic)
